#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

#### Loading the Dataset

In [3]:
df = pd.read_csv("../data/raw/Hotel Reservations.csv")

print("Dataset Shape:", df.shape)

print("\nFirst Five Records:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

Dataset Shape: (36275, 19)

First Five Records:


,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,INN00001,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,2017,10,2,Offline,0,0,0,65.00,0,Not_Canceled
1,INN00002,2,0,2,3,Not Selected,0,Room_Type 1,5,2018,11,6,Online,0,0,0,106.68,1,Not_Canceled
2,INN00003,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,2018,2,28,Online,0,0,0,60.00,0,Canceled
3,INN00004,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,2018,5,20,Online,0,0,0,100.00,0,Canceled
4,INN00005,2,0,1,1,Not Selected,0,Room_Type 1,48,2018,4,11,Online,0,0,0,94.50,0,Canceled



Column Names:
['Booking_ID', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'booking_status']


#### Key Observations
- **Dataset Structure:** The dataset contains 36,275 hotel reservation records and 19 variables, covering guest characteristics, booking details, room preferences, pricing, arrival dates, and reservation outcomes.
- **Analytical Scope:** The available variables support univariate, bivariate, and multivariate exploration, while the arrival-date fields enable temporal analysis of scheduled reservation volumes.
- **Forecasting Considerations:** `Booking_ID` serves as a record identifier, while `booking_status` provides cancellation information. The primary forecasting target will be constructed by aggregating reservations by valid scheduled arrival dates.

#### Preparing Derived Variables for Exploratory Analysis

In [4]:
df["arrival_datetime"] = pd.to_datetime(
    df[["arrival_year", "arrival_month", "arrival_date"]].rename(
        columns={
            "arrival_year": "year",
            "arrival_month": "month",
            "arrival_date": "day"
        }
    ),
    errors="coerce"
)

df["total_nights"] = (
    df["no_of_weekend_nights"] +
    df["no_of_week_nights"]
)

df["total_guests"] = (
    df["no_of_adults"] +
    df["no_of_children"]
)

df["is_canceled"] = (
    df["booking_status"] == "Canceled"
).astype(int)

print("Derived Variables Summary:")
print(
    df[
        [
            "arrival_datetime",
            "total_nights",
            "total_guests",
            "is_canceled"
        ]
    ].head().to_string(index=False)
)

print("\nArrival Date Validation:")
print(f"Valid Arrival Dates: {df['arrival_datetime'].notna().sum():,}")
print(f"Invalid Arrival Dates: {df['arrival_datetime'].isna().sum():,}")

print("\nDerived Numerical Variables:")
print(
    df[
        ["total_nights", "total_guests", "is_canceled"]
    ].describe().round(2).to_string()
)

Derived Variables Summary:
arrival_datetime  total_nights  total_guests  is_canceled
      2017-10-02             3             2            0
      2018-11-06             5             2            0
      2018-02-28             3             1            1
      2018-05-20             2             2            1
      2018-04-11             2             2            1

Arrival Date Validation:
Valid Arrival Dates: 36,238
Invalid Arrival Dates: 37

Derived Numerical Variables:
       total_nights  total_guests  is_canceled
count      36275.00      36275.00     36275.00
mean           3.02          1.95         0.33
std            1.79          0.65         0.47
min            0.00          1.00         0.00
25%            2.00          2.00         0.00
50%            3.00          2.00         0.00
75%            4.00          2.00         1.00
max           24.00         12.00         1.00


#### Key Observations
- **Derived Feature Validation:** Scheduled arrival dates, total nights, total guests, and cancellation indicators have been successfully created. The dataset contains 36,238 valid arrival dates and 37 invalid dates, consistent with the earlier data-quality assessment.
- **Length of Stay:** Reservations average 3.02 nights, with a median of 3 nights and a maximum of 24 nights. The minimum of zero nights warrants further examination during exploratory analysis.
- **Guest Composition:** Reservations average 1.95 guests, with a median of 2 and a maximum of 12, indicating that two-guest reservations are typical.
- **Cancellation Indicator:** The mean cancellation indicator of approximately 0.33 corresponds to an overall cancellation rate of roughly 33%.

## Univariate Analysis

#### Examining Reservation Status Distribution

In [5]:
import plotly.express as px

status_summary = (
    df["booking_status"]
    .value_counts()
    .rename_axis("Booking_Status")
    .reset_index(name="Reservations")
)

status_summary["Percentage"] = (
    status_summary["Reservations"] /
    status_summary["Reservations"].sum() * 100
)

print("Reservation Status Distribution:")
print(status_summary.round(2).to_string(index=False))

fig = px.bar(
    status_summary,
    x="Booking_Status",
    y="Reservations",
    color="Booking_Status",
    text="Reservations",
    title="Distribution of Hotel Reservation Status",
    labels={
        "Booking_Status": "Reservation Status",
        "Reservations": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=900,
    height=500,
    showlegend=False
)

fig.show()

Reservation Status Distribution:
Booking_Status  Reservations  Percentage
  Not_Canceled         24390       67.24
      Canceled         11885       32.76


**Business Insights**

- **The substantial cancellation rate highlights the importance of distinguishing scheduled reservations from realized hotel occupancy when supporting operational planning.**
- **Understanding cancellation patterns across booking characteristics and arrival periods can help identify segments associated with greater reservation uncertainty.**

#### Examining Booking Lead Time Distribution

In [6]:
lead_time_summary = df["lead_time"].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
)

print("Booking Lead Time Summary:")
print(lead_time_summary.round(2).to_string())

print("\nAdditional Lead Time Statistics:")
print(f"Same-Day Reservations: {(df['lead_time'] == 0).sum():,}")
print(f"Reservations Within 7 Days: {(df['lead_time'] <= 7).sum():,}")
print(f"Reservations Beyond 90 Days: {(df['lead_time'] > 90).sum():,}")

fig = px.histogram(
    df,
    x="lead_time",
    nbins=50,
    title="Distribution of Booking Lead Time",
    labels={
        "lead_time": "Booking Lead Time (Days)",
        "count": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_layout(
    width=950,
    height=500,
    xaxis_title="Booking Lead Time (Days)",
    yaxis_title="Number of Reservations",
    showlegend=False
)

fig.show()

Booking Lead Time Summary:
count    36275.00
mean        85.23
std         85.93
min          0.00
5%           1.00
25%         17.00
50%         57.00
75%        126.00
95%        273.00
max        443.00

Additional Lead Time Statistics:
Same-Day Reservations: 1,297
Reservations Within 7 Days: 5,801
Reservations Beyond 90 Days: 13,049


#### Key Observations
- **Booking Lead Time Distribution:** Reservations have an average lead time of 85.23 days and a median of 57 days, indicating a right-skewed distribution with some substantially longer advance bookings.
- **Short-Notice Reservations:** A total of 5,801 reservations (15.99%) are made within seven days of arrival, including 1,297 same-day reservations (3.58%).
- **Advance Booking Behavior:** Approximately 35.97% of reservations (13,049) are made more than 90 days in advance, with lead times extending up to 443 days.

**Business Insights**

- **The wide variation in booking lead times highlights the need to understand both short-notice and advance reservation behavior when assessing hotel booking patterns.**
- **The substantial proportion of reservations made more than 90 days ahead suggests that advance booking activity is an important consideration for reservation planning and demand visibility.**

#### Examining Length of Stay Distribution

In [7]:
stay_summary = df["total_nights"].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
)

print("Length of Stay Summary:")
print(stay_summary.round(2).to_string())

print("\nAdditional Stay Statistics:")
print(f"Zero-Night Reservations: {(df['total_nights'] == 0).sum():,}")
print(f"One-Night Reservations: {(df['total_nights'] == 1).sum():,}")
print(f"Reservations of 2–4 Nights: {df['total_nights'].between(2, 4).sum():,}")
print(f"Reservations Longer Than 7 Nights: {(df['total_nights'] > 7).sum():,}")

print("\nWeekday and Weekend Night Statistics:")
print(
    df[
        ["no_of_week_nights", "no_of_weekend_nights"]
    ].describe().round(2).to_string()
)

stay_distribution = (
    df["total_nights"]
    .value_counts()
    .sort_index()
    .rename_axis("Total_Nights")
    .reset_index(name="Reservations")
)

fig = px.bar(
    stay_distribution,
    x="Total_Nights",
    y="Reservations",
    title="Distribution of Hotel Reservation Length of Stay",
    labels={
        "Total_Nights": "Total Nights",
        "Reservations": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_layout(
    width=950,
    height=500,
    xaxis_title="Length of Stay (Nights)",
    yaxis_title="Number of Reservations",
    showlegend=False
)

fig.show()

Length of Stay Summary:
count    36275.00
mean         3.02
std          1.79
min          0.00
5%           1.00
25%          2.00
50%          3.00
75%          4.00
95%          6.00
max         24.00

Additional Stay Statistics:
Zero-Night Reservations: 78
One-Night Reservations: 6,604
Reservations of 2–4 Nights: 24,417
Reservations Longer Than 7 Nights: 583

Weekday and Weekend Night Statistics:
       no_of_week_nights  no_of_weekend_nights
count           36275.00              36275.00
mean                2.20                  0.81
std                 1.41                  0.87
min                 0.00                  0.00
25%                 1.00                  0.00
50%                 2.00                  1.00
75%                 3.00                  2.00
max                17.00                  7.00


#### Key Observations
- **Length of Stay Distribution:** Reservations average 3.02 nights, with a median of 3 nights. Approximately 67.31% of reservations (24,417) are for 2–4 nights, indicating a concentration of relatively short stays.
- **Weekday and Weekend Stays:** Reservations average 2.20 weekday nights and 0.81 weekend nights, showing a greater contribution from weekdays to the typical booking duration.
- **Unusual Stay Durations:** Only 583 reservations (1.61%) exceed seven nights, while 78 reservations (0.22%) record zero nights. The zero-night records warrant further examination during data preparation.

**Business Insights**

- **The concentration of reservations within 2–4 nights highlights the importance of short-duration stays when evaluating hotel booking patterns and accommodation requirements.**
- **The greater contribution of weekday nights suggests that weekday accommodation requirements are an important consideration when assessing reservation characteristics and operational planning.**

#### Examining Guest Composition

In [9]:
guest_summary = df[
    ["no_of_adults", "no_of_children", "total_guests"]
].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
)

print("Guest Composition Summary:")
print(guest_summary.round(2).to_string())

print("\nAdditional Guest Statistics:")
print(f"Single-Guest Reservations: {(df['total_guests'] == 1).sum():,}")
print(f"Two-Guest Reservations: {(df['total_guests'] == 2).sum():,}")
print(f"Reservations With Children: {(df['no_of_children'] > 0).sum():,}")
print(f"Reservations With No Adults: {(df['no_of_adults'] == 0).sum():,}")
print(f"Reservations With More Than 4 Guests: {(df['total_guests'] > 4).sum():,}")

guest_distribution = (
    df["total_guests"]
    .value_counts()
    .sort_index()
    .rename_axis("Total_Guests")
    .reset_index(name="Reservations")
)

fig = px.bar(
    guest_distribution,
    x="Total_Guests",
    y="Reservations",
    title="Distribution of Guests per Reservation",
    text="Reservations",
    labels={
        "Total_Guests": "Number of Guests",
        "Reservations": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=950,
    height=500,
    xaxis_title="Number of Guests",
    yaxis_title="Number of Reservations",
    showlegend=False
)

fig.show()

Guest Composition Summary:
       no_of_adults  no_of_children  total_guests
count      36275.00        36275.00      36275.00
mean           1.84            0.11          1.95
std            0.52            0.40          0.65
min            0.00            0.00          1.00
5%             1.00            0.00          1.00
25%            2.00            0.00          2.00
50%            2.00            0.00          2.00
75%            2.00            0.00          2.00
95%            3.00            1.00          3.00
max            4.00           10.00         12.00

Additional Guest Statistics:
Single-Guest Reservations: 7,552
Two-Guest Reservations: 23,942
Reservations With Children: 2,698
Reservations With No Adults: 139
Reservations With More Than 4 Guests: 18


#### Key Observations
- **Guest Composition:** Reservations average 1.95 guests, with a median of 2. Two-guest reservations dominate at 66.00% (23,942), followed by single-guest reservations at 20.82% (7,552).
- **Reservations With Children:** Only 7.44% of reservations (2,698) include children, indicating that most bookings involve adults without accompanying children.
- **Unusual Guest Configurations:** The dataset contains 139 reservations with no adults and 18 reservations involving more than four guests. The maximum recorded party size is 12, including up to 10 children, warranting further validation during data preparation.

**Business Insights**

- **The dominance of two-guest reservations highlights the importance of standard double-occupancy accommodation when evaluating customer booking preferences and room requirements.**
- **The relatively small proportion of reservations involving children suggests that family-oriented bookings represent a smaller segment of overall reservation activity.**

#### Examining Room Price Distribution

In [10]:
price_summary = df["avg_price_per_room"].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
)

print("Room Price Summary:")
print(price_summary.round(2).to_string())

print("\nAdditional Pricing Statistics:")
print(f"Zero-Price Reservations: {(df['avg_price_per_room'] == 0).sum():,}")
print(f"Reservations Below 50: {(df['avg_price_per_room'] < 50).sum():,}")
print(f"Reservations Above 150: {(df['avg_price_per_room'] > 150).sum():,}")
print(f"Reservations Above 200: {(df['avg_price_per_room'] > 200).sum():,}")

fig = px.histogram(
    df,
    x="avg_price_per_room",
    nbins=50,
    title="Distribution of Average Room Price",
    template="plotly_white",
    labels={
        "avg_price_per_room": "Average Price per Room"
    }
)

fig.update_layout(
    width=950,
    height=500,
    xaxis_title="Average Price per Room",
    yaxis_title="Number of Reservations",
    showlegend=False
)

fig.show()

Room Price Summary:
count    36275.00
mean       103.42
std         35.09
min          0.00
5%          61.00
25%         80.30
50%         99.45
75%        120.00
95%        165.00
max        540.00

Additional Pricing Statistics:
Zero-Price Reservations: 545
Reservations Below 50: 896
Reservations Above 150: 3,220
Reservations Above 200: 496


#### Key Observations
- **Room Price Distribution:** The average room price is 103.42, with a median of 99.45. The middle 50% of reservations fall between 80.30 and 120.00, indicating a concentration around moderate price levels.
- **Higher-Priced Reservations:** Approximately 8.88% of reservations (3,220) have room prices above 150, while 496 reservations (1.37%) exceed 200. The maximum recorded price is 540.
- **Unusual Pricing Records:** The dataset contains 545 zero-price reservations (1.50%) and 896 reservations (2.47%) priced below 50. These records warrant further investigation before determining whether any preprocessing is necessary.

**Business Insights**

- **The concentration of room prices around the median suggests that a relatively narrow pricing range accounts for a substantial share of hotel reservations.**
- **The presence of premium-priced and zero-price reservations highlights the importance of examining pricing differences across room types, market segments, and reservation outcomes.**

#### Examining Market Segment Distribution

In [11]:
segment_summary = (
    df["market_segment_type"]
    .value_counts()
    .rename_axis("Market_Segment")
    .reset_index(name="Reservations")
)

segment_summary["Percentage"] = (
    segment_summary["Reservations"] /
    segment_summary["Reservations"].sum() * 100
)

print("Market Segment Distribution:")
print(segment_summary.round(2).to_string(index=False))

fig = px.bar(
    segment_summary,
    x="Market_Segment",
    y="Reservations",
    color="Market_Segment",
    text="Reservations",
    title="Distribution of Reservations by Market Segment",
    labels={
        "Market_Segment": "Market Segment",
        "Reservations": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Market Segment",
    yaxis_title="Number of Reservations"
)

fig.show()

Market Segment Distribution:
Market_Segment  Reservations  Percentage
        Online         23214       63.99
       Offline         10528       29.02
     Corporate          2017        5.56
 Complementary           391        1.08
      Aviation           125        0.34


#### Key Observations
- **Online Booking Dominance:** Online reservations account for 63.99% (23,214) of all bookings, making this the largest market segment.
- **Offline Booking Contribution:** Offline reservations represent 29.02% (10,528), with Online and Offline channels collectively accounting for approximately 93.01% of total reservations.
- **Smaller Market Segments:** Corporate bookings contribute 5.56%, while Complementary and Aviation segments together represent approximately 1.42% of reservations.

**Business Insights**

- **The strong concentration of reservations in Online and Offline channels highlights their importance in understanding overall booking activity and customer acquisition patterns.**
- **The smaller Corporate, Complementary, and Aviation segments warrant separate evaluation to identify differences in booking behavior without allowing their limited representation to distort overall findings.**

#### Examining Room Type Distribution

In [12]:
room_summary = (
    df["room_type_reserved"]
    .value_counts()
    .rename_axis("Room_Type")
    .reset_index(name="Reservations")
)

room_summary["Percentage"] = (
    room_summary["Reservations"] /
    room_summary["Reservations"].sum() * 100
)

print("Room Type Distribution:")
print(room_summary.round(2).to_string(index=False))

fig = px.bar(
    room_summary,
    x="Room_Type",
    y="Reservations",
    color="Room_Type",
    text="Reservations",
    title="Distribution of Reservations by Room Type",
    labels={
        "Room_Type": "Room Type",
        "Reservations": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Room Type",
    yaxis_title="Number of Reservations"
)

fig.show()

Room Type Distribution:
  Room_Type  Reservations  Percentage
Room_Type 1         28130       77.55
Room_Type 4          6057       16.70
Room_Type 6           966        2.66
Room_Type 2           692        1.91
Room_Type 5           265        0.73
Room_Type 7           158        0.44
Room_Type 3             7        0.02


#### Key Observations
- **Dominant Room Category:** Room_Type 1 accounts for 77.55% (28,130) of all reservations, making it the most frequently selected room category.
- **Secondary Room Preference:** Room_Type 4 represents 16.70% (6,057) of reservations. Together, Room_Type 1 and Room_Type 4 account for 94.25% of total bookings.
- **Limited Representation of Other Rooms:** The remaining five room types collectively contribute approximately 5.75% of reservations, with Room_Type 3 appearing in only seven records.

**Business Insights**

- **The substantial concentration of reservations in Room_Type 1 and Room_Type 4 highlights their importance when analyzing room demand and accommodation preferences.**
- **The limited representation of other room categories suggests that room-specific comparisons should account for sample size differences to avoid drawing conclusions from sparsely represented categories.**

#### Examining Meal Plan Distribution

In [13]:
meal_summary = (
    df["type_of_meal_plan"]
    .value_counts()
    .rename_axis("Meal_Plan")
    .reset_index(name="Reservations")
)

meal_summary["Percentage"] = (
    meal_summary["Reservations"] /
    meal_summary["Reservations"].sum() * 100
)

print("Meal Plan Distribution:")
print(meal_summary.round(2).to_string(index=False))

fig = px.bar(
    meal_summary,
    x="Meal_Plan",
    y="Reservations",
    color="Meal_Plan",
    text="Reservations",
    title="Distribution of Reservations by Meal Plan",
    labels={
        "Meal_Plan": "Meal Plan",
        "Reservations": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Meal Plan",
    yaxis_title="Number of Reservations"
)

fig.show()

Meal Plan Distribution:
   Meal_Plan  Reservations  Percentage
 Meal Plan 1         27835       76.73
Not Selected          5130       14.14
 Meal Plan 2          3305        9.11
 Meal Plan 3             5        0.01


#### Key Observations
- **Dominant Meal Plan:** Meal Plan 1 accounts for 76.73% (27,835) of all reservations, making it the most frequently selected meal plan.
- **Alternative Preferences:** Reservations without a selected meal plan represent 14.14% (5,130), while Meal Plan 2 accounts for 9.11% (3,305).
- **Rare Meal Plan Category:** Meal Plan 3 appears in only five reservations (approximately 0.01%), indicating extremely limited representation.

**Business Insights**

- **The strong preference for Meal Plan 1 highlights its importance when examining customer reservation choices and meal-service requirements.**
- **The presence of reservations without selected meal plans and the limited representation of alternative plans suggest that meal-plan comparisons should account for differences in category size.**

#### Examining Repeat Guest Distribution

In [14]:
repeat_guest_summary = (
    df["repeated_guest"]
    .map({0: "First-Time Guest", 1: "Repeat Guest"})
    .value_counts()
    .rename_axis("Guest_Type")
    .reset_index(name="Reservations")
)

repeat_guest_summary["Percentage"] = (
    repeat_guest_summary["Reservations"] /
    repeat_guest_summary["Reservations"].sum() * 100
)

print("Repeat Guest Distribution:")
print(repeat_guest_summary.round(2).to_string(index=False))

fig = px.bar(
    repeat_guest_summary,
    x="Guest_Type",
    y="Reservations",
    color="Guest_Type",
    text="Reservations",
    title="Distribution of First-Time and Repeat Guest Reservations",
    labels={
        "Guest_Type": "Guest Type",
        "Reservations": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=900,
    height=500,
    showlegend=False,
    xaxis_title="Guest Type",
    yaxis_title="Number of Reservations"
)

fig.show()

Repeat Guest Distribution:
      Guest_Type  Reservations  Percentage
First-Time Guest         35345       97.44
    Repeat Guest           930        2.56


#### Key Observations
- **First-Time Guest Dominance:** First-time guest reservations account for 97.44% (35,345) of all bookings, representing the overwhelming majority of reservation activity.
- **Limited Repeat Guest Representation:** Repeat guests account for only 2.56% (930) of reservations, indicating that previously recorded guests represent a relatively small portion of the dataset.

**Business Insights**

- **The strong concentration of first-time guest reservations highlights the importance of new-customer booking activity when understanding overall reservation patterns.**
- **The limited representation of repeat guests suggests that loyalty-related comparisons should be interpreted carefully, particularly when examining cancellation behavior and booking preferences.**

#### Examining Special Request Distribution

In [15]:
request_summary = (
    df["no_of_special_requests"]
    .value_counts()
    .sort_index()
    .rename_axis("Special_Requests")
    .reset_index(name="Reservations")
)

request_summary["Percentage"] = (
    request_summary["Reservations"] /
    request_summary["Reservations"].sum() * 100
)

print("Special Request Distribution:")
print(request_summary.round(2).to_string(index=False))

print("\nAdditional Special Request Statistics:")
print(f"Average Special Requests: {df['no_of_special_requests'].mean():.2f}")
print(f"Reservations Without Requests: {(df['no_of_special_requests'] == 0).sum():,}")
print(f"Reservations With Requests: {(df['no_of_special_requests'] > 0).sum():,}")
print(f"Maximum Special Requests: {df['no_of_special_requests'].max()}")

fig = px.bar(
    request_summary,
    x="Special_Requests",
    y="Reservations",
    text="Reservations",
    title="Distribution of Special Requests per Reservation",
    labels={
        "Special_Requests": "Number of Special Requests",
        "Reservations": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=950,
    height=500,
    xaxis_title="Number of Special Requests",
    yaxis_title="Number of Reservations",
    showlegend=False
)

fig.show()

Special Request Distribution:
 Special_Requests  Reservations  Percentage
                0         19777       54.52
                1         11373       31.35
                2          4364       12.03
                3           675        1.86
                4            78        0.22
                5             8        0.02

Additional Special Request Statistics:
Average Special Requests: 0.62
Reservations Without Requests: 19,777
Reservations With Requests: 16,498
Maximum Special Requests: 5


#### Key Observations
- **Reservations Without Special Requests:** A majority of reservations (54.52%, or 19,777) contain no special requests, while 45.48% (16,498) include at least one.
- **Special Request Frequency:** Reservations with one special request account for 31.35%, followed by two requests at 12.03%. The average is 0.62 requests per reservation.
- **Multiple Special Requests:** Only 2.10% of reservations contain three or more special requests, with a maximum of five, indicating that higher request counts are relatively uncommon.

**Business Insights**

- **The substantial proportion of reservations containing special requests highlights the importance of accommodating individual guest preferences in hotel service planning.**
- **The concentration of reservations within zero to two special requests suggests that most guest-specific service requirements involve relatively few requests per booking.**

#### Examining Car Parking Requirements

In [16]:
parking_summary = (
    df["required_car_parking_space"]
    .map({0: "Not Required", 1: "Required"})
    .value_counts()
    .rename_axis("Parking_Requirement")
    .reset_index(name="Reservations")
)

parking_summary["Percentage"] = (
    parking_summary["Reservations"] /
    parking_summary["Reservations"].sum() * 100
)

print("Car Parking Requirement Distribution:")
print(parking_summary.round(2).to_string(index=False))

fig = px.bar(
    parking_summary,
    x="Parking_Requirement",
    y="Reservations",
    color="Parking_Requirement",
    text="Reservations",
    title="Distribution of Car Parking Requirements",
    labels={
        "Parking_Requirement": "Car Parking Requirement",
        "Reservations": "Number of Reservations"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=900,
    height=500,
    showlegend=False,
    xaxis_title="Car Parking Requirement",
    yaxis_title="Number of Reservations"
)

fig.show()

Car Parking Requirement Distribution:
Parking_Requirement  Reservations  Percentage
       Not Required         35151        96.9
           Required          1124         3.1


#### Examining Previous Booking History

In [17]:
history_columns = [
    "no_of_previous_cancellations",
    "no_of_previous_bookings_not_canceled"
]

print("Previous Booking History Summary:")
print(
    df[history_columns]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.95])
    .round(2)
    .to_string()
)

print("\nAdditional Booking History Statistics:")
print(
    f"Reservations With Previous Cancellations: "
    f"{(df['no_of_previous_cancellations'] > 0).sum():,}"
)
print(
    f"Reservations With Previous Non-Canceled Bookings: "
    f"{(df['no_of_previous_bookings_not_canceled'] > 0).sum():,}"
)
print(
    f"Reservations With Any Recorded Booking History: "
    f"{((df[history_columns].sum(axis=1)) > 0).sum():,}"
)

history_summary = pd.DataFrame({
    "Booking_History": [
        "Previous Cancellations",
        "Previous Non-Canceled Bookings"
    ],
    "Reservations": [
        (df["no_of_previous_cancellations"] > 0).sum(),
        (df["no_of_previous_bookings_not_canceled"] > 0).sum()
    ]
})

fig = px.bar(
    history_summary,
    x="Booking_History",
    y="Reservations",
    color="Booking_History",
    text="Reservations",
    title="Reservations With Recorded Previous Booking Activity",
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Previous Booking Activity",
    yaxis_title="Number of Reservations"
)

fig.show()

Previous Booking History Summary:
       no_of_previous_cancellations  no_of_previous_bookings_not_canceled
count                      36275.00                              36275.00
mean                           0.02                                  0.15
std                            0.37                                  1.75
min                            0.00                                  0.00
25%                            0.00                                  0.00
50%                            0.00                                  0.00
75%                            0.00                                  0.00
95%                            0.00                                  0.00
max                           13.00                                 58.00

Additional Booking History Statistics:
Reservations With Previous Cancellations: 338
Reservations With Previous Non-Canceled Bookings: 812
Reservations With Any Recorded Booking History: 930


#### Examining Previous Booking History

In [18]:
history_columns = [
    "no_of_previous_cancellations",
    "no_of_previous_bookings_not_canceled"
]

print("Previous Booking History Summary:")
print(
    df[history_columns]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.95])
    .round(2)
    .to_string()
)

print("\nAdditional Booking History Statistics:")
print(
    f"Reservations With Previous Cancellations: "
    f"{(df['no_of_previous_cancellations'] > 0).sum():,}"
)
print(
    f"Reservations With Previous Non-Canceled Bookings: "
    f"{(df['no_of_previous_bookings_not_canceled'] > 0).sum():,}"
)
print(
    f"Reservations With Any Recorded Booking History: "
    f"{((df[history_columns].sum(axis=1)) > 0).sum():,}"
)

history_summary = pd.DataFrame({
    "Booking_History": [
        "Previous Cancellations",
        "Previous Non-Canceled Bookings"
    ],
    "Reservations": [
        (df["no_of_previous_cancellations"] > 0).sum(),
        (df["no_of_previous_bookings_not_canceled"] > 0).sum()
    ]
})

fig = px.bar(
    history_summary,
    x="Booking_History",
    y="Reservations",
    color="Booking_History",
    text="Reservations",
    title="Reservations With Recorded Previous Booking Activity",
    template="plotly_white"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Previous Booking Activity",
    yaxis_title="Number of Reservations"
)

fig.show()

Previous Booking History Summary:
       no_of_previous_cancellations  no_of_previous_bookings_not_canceled
count                      36275.00                              36275.00
mean                           0.02                                  0.15
std                            0.37                                  1.75
min                            0.00                                  0.00
25%                            0.00                                  0.00
50%                            0.00                                  0.00
75%                            0.00                                  0.00
95%                            0.00                                  0.00
max                           13.00                                 58.00

Additional Booking History Statistics:
Reservations With Previous Cancellations: 338
Reservations With Previous Non-Canceled Bookings: 812
Reservations With Any Recorded Booking History: 930


#### Key Observations
- **Limited Previous Booking Activity:** Only 930 reservations (2.56%) have any recorded booking history, indicating that the overwhelming majority have no previous booking activity recorded.
- **Previous Cancellation History:** Only 338 reservations (0.93%) have previous cancellations, with a maximum of 13 cancellations recorded for an individual reservation.
- **Previous Non-Canceled Bookings:** A total of 812 reservations (2.24%) have previously completed non-canceled bookings, with historical counts reaching 58. Both historical variables have a median of zero, reflecting highly concentrated activity.

**Business Insights**

- **The limited representation of customers with recorded booking history suggests that historical customer behavior provides information for only a small segment of reservations.**
- **The concentration of previous booking activity among relatively few customers highlights the importance of examining repeat-guest behavior separately when evaluating cancellation patterns and customer retention.**

## Bivariate Analysis

#### Examining Cancellation Rates Across Booking Lead Times

In [19]:
lead_time_bins = [-1, 7, 30, 90, 180, float("inf")]

lead_time_labels = [
    "0–7 Days",
    "8–30 Days",
    "31–90 Days",
    "91–180 Days",
    "181+ Days"
]

df["lead_time_group"] = pd.cut(
    df["lead_time"],
    bins=lead_time_bins,
    labels=lead_time_labels
)

lead_time_cancellation = (
    df.groupby("lead_time_group", observed=True)
    .agg(
        Reservations=("Booking_ID", "count"),
        Cancellations=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .reset_index()
)

lead_time_cancellation["Cancellation_Rate"] *= 100

print("Cancellation Rates by Booking Lead Time:")
print(lead_time_cancellation.round(2).to_string(index=False))

fig = px.bar(
    lead_time_cancellation,
    x="lead_time_group",
    y="Cancellation_Rate",
    color="lead_time_group",
    text="Cancellation_Rate",
    title="Cancellation Rate by Booking Lead Time",
    labels={
        "lead_time_group": "Booking Lead Time",
        "Cancellation_Rate": "Cancellation Rate (%)"
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Booking Lead Time",
    yaxis_title="Cancellation Rate (%)",
    yaxis_range=[0, 100]
)

fig.show()

Cancellation Rates by Booking Lead Time:
lead_time_group  Reservations  Cancellations  Cancellation_Rate
       0–7 Days          5801            514               8.86
      8–30 Days          6610           1249              18.90
     31–90 Days         10815           2733              25.27
    91–180 Days          7772           3490              44.90
      181+ Days          5277           3899              73.89


#### Key Observations
- **Increasing Cancellation Rates:** Cancellation rates rise consistently with booking lead time, from 8.86% for reservations made within 7 days to 73.89% for reservations made more than 180 days in advance.
- **High Cancellation Exposure:** Reservations made 91–180 days ahead have a cancellation rate of 44.90%, increasing to 73.89% for bookings made more than 180 days ahead.
- **Short-Notice Booking Stability:** Reservations made within 7 days have the lowest cancellation rate (8.86%), compared with 18.90% for bookings made 8–30 days ahead.

**Business Insights**

- **The strong association between longer booking lead times and higher cancellation rates highlights advance reservations as an important source of uncertainty in hotel reservation planning.**
- **Short-notice reservations exhibit substantially lower cancellation rates, suggesting that booking lead time may be valuable for assessing cancellation exposure and interpreting future scheduled reservation volumes.**

#### Examining Cancellation Rates Across Market Segments

In [20]:
segment_cancellation = (
    df.groupby("market_segment_type")
    .agg(
        Reservations=("Booking_ID", "count"),
        Cancellations=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .reset_index()
    .sort_values("Cancellation_Rate", ascending=False)
)

segment_cancellation["Cancellation_Rate"] *= 100

print("Cancellation Rates by Market Segment:")
print(segment_cancellation.round(2).to_string(index=False))

fig = px.bar(
    segment_cancellation,
    x="market_segment_type",
    y="Cancellation_Rate",
    color="market_segment_type",
    text="Cancellation_Rate",
    title="Cancellation Rate by Market Segment",
    labels={
        "market_segment_type": "Market Segment",
        "Cancellation_Rate": "Cancellation Rate (%)"
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Market Segment",
    yaxis_title="Cancellation Rate (%)",
    yaxis_range=[0, 100]
)

fig.show()

Cancellation Rates by Market Segment:
market_segment_type  Reservations  Cancellations  Cancellation_Rate
             Online         23214           8475              36.51
            Offline         10528           3153              29.95
           Aviation           125             37              29.60
          Corporate          2017            220              10.91
      Complementary           391              0               0.00


#### Key Observations
- **Online Cancellation Exposure:** Online reservations have the highest cancellation rate at 36.51% (8,475 cancellations), compared with 29.95% for Offline reservations (3,153 cancellations).
- **Corporate Booking Stability:** Corporate reservations have a considerably lower cancellation rate of 10.91%, despite accounting for 2,017 reservations.
- **Smaller Market Segments:** Aviation reservations have a cancellation rate of 29.60%, while Complementary reservations record no cancellations. These findings should be interpreted considering their relatively small sample sizes of 125 and 391 reservations, respectively.

**Business Insights**

- **The higher cancellation rate among Online reservations highlights this channel as an important area for understanding reservation uncertainty and potential fluctuations in expected hotel demand.**
- **The comparatively low cancellation rate among Corporate reservations suggests greater booking stability within this segment, although differences in customer characteristics and booking conditions may also contribute to the observed pattern.**

#### Examining Cancellation Rates Across Room Types

In [21]:
room_cancellation = (
    df.groupby("room_type_reserved")
    .agg(
        Reservations=("Booking_ID", "count"),
        Cancellations=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .reset_index()
    .sort_values("Cancellation_Rate", ascending=False)
)

room_cancellation["Cancellation_Rate"] *= 100

print("Cancellation Rates by Room Type:")
print(room_cancellation.round(2).to_string(index=False))

fig = px.bar(
    room_cancellation,
    x="room_type_reserved",
    y="Cancellation_Rate",
    color="room_type_reserved",
    text="Cancellation_Rate",
    title="Cancellation Rate by Room Type",
    labels={
        "room_type_reserved": "Room Type",
        "Cancellation_Rate": "Cancellation Rate (%)"
    },
    template="plotly_white",
    hover_data={
        "Reservations": True,
        "Cancellations": True,
        "Cancellation_Rate": ":.2f"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Room Type",
    yaxis_title="Cancellation Rate (%)",
    yaxis_range=[0, 100]
)

fig.show()

Cancellation Rates by Room Type:
room_type_reserved  Reservations  Cancellations  Cancellation_Rate
       Room_Type 6           966            406              42.03
       Room_Type 4          6057           2069              34.16
       Room_Type 2           692            228              32.95
       Room_Type 1         28130           9072              32.25
       Room_Type 3             7              2              28.57
       Room_Type 5           265             72              27.17
       Room_Type 7           158             36              22.78


#### Key Observations
- **Highest Cancellation Rate:** Room_Type 6 records the highest cancellation rate at 42.03% (406 of 966 reservations), exceeding the overall cancellation rate of 32.76%.
- **Dominant Room Categories:** Room_Type 1, representing the majority of reservations, has a cancellation rate of 32.25%, while Room_Type 4 records 34.16%, indicating relatively similar cancellation behavior across the two largest categories.
- **Variation Across Smaller Categories:** Room_Type 7 has the lowest observed cancellation rate at 22.78%, while Room_Type 3 records 28.57% across only seven reservations, making its estimated rate particularly unreliable.

**Business Insights**

- **The elevated cancellation rate for Room_Type 6 suggests that this category warrants further investigation to understand its reservation characteristics and potential cancellation exposure.**
- **The relatively similar cancellation rates of Room_Type 1 and Room_Type 4 indicate that the two dominant room categories contribute to overall cancellation patterns without showing substantial differences in their individual cancellation rates.**

#### Examining Cancellation Rates Across Lengths of Stay

In [22]:
stay_bins = [-1, 1, 2, 4, 7, float("inf")]

stay_labels = [
    "0–1 Nights",
    "2 Nights",
    "3–4 Nights",
    "5–7 Nights",
    "8+ Nights"
]

df["stay_length_group"] = pd.cut(
    df["total_nights"],
    bins=stay_bins,
    labels=stay_labels
)

stay_cancellation = (
    df.groupby("stay_length_group", observed=True)
    .agg(
        Reservations=("Booking_ID", "count"),
        Cancellations=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .reset_index()
)

stay_cancellation["Cancellation_Rate"] *= 100

print("Cancellation Rates by Length of Stay:")
print(stay_cancellation.round(2).to_string(index=False))

fig = px.bar(
    stay_cancellation,
    x="stay_length_group",
    y="Cancellation_Rate",
    color="stay_length_group",
    text="Cancellation_Rate",
    title="Cancellation Rate by Length of Stay",
    labels={
        "stay_length_group": "Length of Stay",
        "Cancellation_Rate": "Cancellation Rate (%)"
    },
    template="plotly_white",
    hover_data={
        "Reservations": True,
        "Cancellations": True
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Length of Stay",
    yaxis_title="Cancellation Rate (%)",
    yaxis_range=[0, 100]
)

fig.show()

Cancellation Rates by Length of Stay:
stay_length_group  Reservations  Cancellations  Cancellation_Rate
       0–1 Nights          6682           1468              21.97
         2 Nights          8472           2899              34.22
       3–4 Nights         15945           5527              34.66
       5–7 Nights          4593           1671              36.38
        8+ Nights           583            320              54.89


#### Key Observations
- **Higher Cancellation Rates for Longer Stays:** Cancellation rates increase from 21.97% for reservations lasting 0–1 nights to 54.89% for stays exceeding seven nights.
- **Moderate-Length Stay Patterns:** Reservations lasting 2–7 nights exhibit relatively similar cancellation rates, ranging from 34.22% to 36.38%.
- **Extended-Stay Cancellation Exposure:** Reservations exceeding seven nights have the highest cancellation rate at 54.89%, although they represent only 583 reservations.

**Business Insights**

- **The higher cancellation rates associated with extended stays suggest that reservation duration may be an important factor when assessing booking uncertainty and potential changes in expected room demand.**
- **Shorter reservations exhibit comparatively lower cancellation rates, while the elevated cancellation exposure among extended stays highlights the value of considering stay duration when analyzing reservation reliability.**

#### Examining Cancellation Rates Across Room Price Ranges

In [23]:
price_bins = [-0.01, 50, 80, 120, 150, float("inf")]

price_labels = [
    "0–50",
    "50–80",
    "80–120",
    "120–150",
    "150+"
]

df["price_group"] = pd.cut(
    df["avg_price_per_room"],
    bins=price_bins,
    labels=price_labels,
    include_lowest=True
)

price_cancellation = (
    df.groupby("price_group", observed=True)
    .agg(
        Reservations=("Booking_ID", "count"),
        Cancellations=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .reset_index()
)

price_cancellation["Cancellation_Rate"] *= 100

print("Cancellation Rates by Room Price Range:")
print(price_cancellation.round(2).to_string(index=False))

fig = px.bar(
    price_cancellation,
    x="price_group",
    y="Cancellation_Rate",
    color="price_group",
    text="Cancellation_Rate",
    title="Cancellation Rate by Room Price Range",
    labels={
        "price_group": "Average Room Price Range",
        "Cancellation_Rate": "Cancellation Rate (%)"
    },
    template="plotly_white",
    hover_data={
        "Reservations": True,
        "Cancellations": True
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Average Room Price Range",
    yaxis_title="Cancellation Rate (%)",
    yaxis_range=[0, 100]
)

fig.show()

Cancellation Rates by Room Price Range:
price_group  Reservations  Cancellations  Cancellation_Rate
       0–50           909             43               4.73
      50–80          8028           1877              23.38
     80–120         18280           6581              36.00
    120–150          5838           2226              38.13
       150+          3220           1158              35.96


#### Key Observations
- **Lowest Cancellation Rate:** Reservations priced between 0–50 have the lowest cancellation rate at 4.73%, compared with 23.38% for reservations priced between 50–80.
- **Higher Cancellation Exposure:** Cancellation rates increase to 36.00% for reservations priced between 80–120 and peak at 38.13% for the 120–150 price range.
- **Nonlinear Pricing Relationship:** Reservations priced above 150 have a slightly lower cancellation rate of 35.96%, indicating that cancellation rates do not increase continuously with room price.

**Business Insights**

- **The higher cancellation rates across mid-to-high room price ranges suggest that pricing may be associated with reservation uncertainty, making room price a useful variable for examining cancellation patterns.**
- **The exceptionally low cancellation rate in the lowest price range warrants further investigation into booking characteristics, including complementary reservations and market segments, before attributing the difference to pricing alone.**

#### Examining Cancellation Rates by Special Requests

In [24]:
request_cancellation = (
    df.groupby("no_of_special_requests")
    .agg(
        Reservations=("Booking_ID", "count"),
        Cancellations=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .reset_index()
)

request_cancellation["Cancellation_Rate"] *= 100

print("Cancellation Rates by Number of Special Requests:")
print(request_cancellation.round(2).to_string(index=False))

fig = px.bar(
    request_cancellation,
    x="no_of_special_requests",
    y="Cancellation_Rate",
    color="no_of_special_requests",
    text="Cancellation_Rate",
    title="Cancellation Rate by Number of Special Requests",
    labels={
        "no_of_special_requests": "Number of Special Requests",
        "Cancellation_Rate": "Cancellation Rate (%)"
    },
    template="plotly_white",
    hover_data={
        "Reservations": True,
        "Cancellations": True
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    width=950,
    height=500,
    showlegend=False,
    xaxis_title="Number of Special Requests",
    yaxis_title="Cancellation Rate (%)",
    yaxis_range=[0, 100],
    xaxis=dict(
        tickmode="linear",
        dtick=1
    )
)

fig.show()

Cancellation Rates by Number of Special Requests:
 no_of_special_requests  Reservations  Cancellations  Cancellation_Rate
                      0         19777           8545              43.21
                      1         11373           2703              23.77
                      2          4364            637              14.60
                      3           675              0               0.00
                      4            78              0               0.00
                      5             8              0               0.00


#### Key Observations
- **Highest Cancellation Rate Without Requests:** Reservations with no special requests have the highest cancellation rate at 43.21%, accounting for 8,545 cancellations across 19,777 reservations.
- **Declining Cancellation Rates:** Cancellation rates decrease to 23.77% for reservations with one special request and 14.60% for those with two requests.
- **No Recorded Cancellations for Three or More Requests:** Reservations with three, four, or five special requests record no cancellations, although these groups contain relatively few observations (675, 78, and 8 reservations, respectively).

**Business Insights**

- **The strong association between special requests and lower cancellation rates suggests that guest-specific requirements may provide useful information when assessing reservation reliability.**
- **Reservations without special requests represent a substantial source of cancellation exposure, highlighting the potential value of incorporating special-request activity into reservation risk analysis while avoiding assumptions of causality.**

## Multivariate Analysis

#### Examining Cancellation Rates Across Lead Times and Market Segments

In [25]:
lead_segment_summary = (
    df.groupby(
        ["lead_time_group", "market_segment_type"],
        observed=True
    )
    .agg(
        Reservations=("Booking_ID", "count"),
        Cancellations=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .reset_index()
)

lead_segment_summary["Cancellation_Rate"] *= 100

print("Cancellation Rates by Lead Time and Market Segment:")
print(
    lead_segment_summary
    .round(2)
    .to_string(index=False)
)

cancellation_matrix = lead_segment_summary.pivot(
    index="market_segment_type",
    columns="lead_time_group",
    values="Cancellation_Rate"
)

reservation_matrix = lead_segment_summary.pivot(
    index="market_segment_type",
    columns="lead_time_group",
    values="Reservations"
)

print("\nReservation Counts by Lead Time and Market Segment:")
print(reservation_matrix.to_string())

fig = px.imshow(
    cancellation_matrix,
    text_auto=".1f",
    color_continuous_scale="Blues",
    aspect="auto",
    title="Cancellation Rates by Booking Lead Time and Market Segment",
    labels={
        "x": "Booking Lead Time",
        "y": "Market Segment",
        "color": "Cancellation Rate (%)"
    },
    zmin=0,
    zmax=100
)

fig.update_layout(
    width=1050,
    height=550,
    template="plotly_white"
)

fig.show()

Cancellation Rates by Lead Time and Market Segment:
lead_time_group market_segment_type  Reservations  Cancellations  Cancellation_Rate
       0–7 Days            Aviation            94             21              22.34
       0–7 Days       Complementary           291              0               0.00
       0–7 Days           Corporate          1007             58               5.76
       0–7 Days             Offline           817             63               7.71
       0–7 Days              Online          3592            372              10.36
      8–30 Days            Aviation            31             16              51.61
      8–30 Days       Complementary            59              0               0.00
      8–30 Days           Corporate           605             54               8.93
      8–30 Days             Offline          1194             32               2.68
      8–30 Days              Online          4721           1147              24.30
     31–90 Days       Co

#### Key Observations
- **High Cancellation Exposure Among Advance Online Reservations:** Online cancellation rates increase from 10.36% for bookings made within seven days to 81.04% for reservations made more than 180 days in advance.
- **Increasing Cancellation Rates Across Major Segments:** Offline cancellation rates rise from 7.71% for bookings made within seven days to 67.62% for bookings made more than 180 days ahead. Corporate reservations show a similar overall pattern, although the longest lead-time group contains only 19 reservations.
- **Differences Between Booking Channels:** Online reservations consistently exhibit higher cancellation rates than Offline reservations across all five lead-time groups, highlighting substantial variation in cancellation behavior between the two dominant market segments.

**Business Insights**

- **The combination of booking lead time and market segment provides a more detailed understanding of cancellation exposure, particularly for Online reservations made far in advance.**
- **The substantial differences between Online and Offline cancellation rates suggest that reservation planning and cancellation-risk assessment may benefit from considering booking channels alongside advance booking periods.**

#### Examining Cancellation Rates Across Lead Times and Room Price Ranges

In [26]:
lead_price_summary = (
    df.groupby(
        ["lead_time_group", "price_group"],
        observed=True
    )
    .agg(
        Reservations=("Booking_ID", "count"),
        Cancellations=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .reset_index()
)

lead_price_summary["Cancellation_Rate"] *= 100

print("Cancellation Rates by Lead Time and Room Price Range:")
print(
    lead_price_summary
    .round(2)
    .to_string(index=False)
)

cancellation_matrix = lead_price_summary.pivot(
    index="price_group",
    columns="lead_time_group",
    values="Cancellation_Rate"
)

reservation_matrix = lead_price_summary.pivot(
    index="price_group",
    columns="lead_time_group",
    values="Reservations"
)

print("\nReservation Counts by Lead Time and Room Price Range:")
print(reservation_matrix.to_string())

fig = px.imshow(
    cancellation_matrix,
    text_auto=".1f",
    color_continuous_scale="Blues",
    aspect="auto",
    title="Cancellation Rates by Booking Lead Time and Room Price Range",
    labels={
        "x": "Booking Lead Time",
        "y": "Room Price Range",
        "color": "Cancellation Rate (%)"
    },
    zmin=0,
    zmax=100
)

fig.update_layout(
    width=1050,
    height=550,
    template="plotly_white"
)

fig.show()

Cancellation Rates by Lead Time and Room Price Range:
lead_time_group price_group  Reservations  Cancellations  Cancellation_Rate
       0–7 Days        0–50           470              2               0.43
       0–7 Days       50–80          1416            129               9.11
       0–7 Days      80–120          2298            179               7.79
       0–7 Days     120–150           915            107              11.69
       0–7 Days        150+           702             97              13.82
      8–30 Days        0–50           171              7               4.09
      8–30 Days       50–80          1261            134              10.63
      8–30 Days      80–120          2982            495              16.60
      8–30 Days     120–150          1317            349              26.50
      8–30 Days        150+           879            264              30.03
     31–90 Days        0–50           152             15               9.87
     31–90 Days       50–80       

#### Key Observations
- **Increasing Cancellation Rates Across Lead Times:** Cancellation rates generally increase with booking lead time across room price ranges. For reservations priced between 80–120, cancellation rates rise from 7.79% within seven days to 79.41% beyond 180 days.
- **Higher Cancellation Rates at Elevated Prices:** Within the 31–90-day lead-time group, cancellation rates increase from 9.87% for the 0–50 price range to 37.66% for reservations priced above 150.
- **Combined Cancellation Exposure:** Reservations priced above 150 and booked 91–180 days ahead have a cancellation rate of 58.66%, compared with 13.82% for similarly priced reservations booked within seven days.

**Business Insights**

- **The combined relationship between booking lead time, room price, and cancellation behavior suggests that evaluating pricing and advance booking periods together provides a more informative assessment of reservation uncertainty.**
- **Higher-priced reservations made further in advance exhibit substantial cancellation exposure, highlighting the potential value of considering both variables when assessing expected reservation volumes and cancellation risk.**

#### Examining Cancellation Rates Across Market Segments and Special Requests

In [27]:
request_segment_summary = (
    df.groupby(
        ["market_segment_type", "no_of_special_requests"]
    )
    .agg(
        Reservations=("Booking_ID", "count"),
        Cancellations=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .reset_index()
)

request_segment_summary["Cancellation_Rate"] *= 100

print("Cancellation Rates by Market Segment and Special Requests:")
print(
    request_segment_summary
    .round(2)
    .to_string(index=False)
)

cancellation_matrix = request_segment_summary.pivot(
    index="market_segment_type",
    columns="no_of_special_requests",
    values="Cancellation_Rate"
)

reservation_matrix = request_segment_summary.pivot(
    index="market_segment_type",
    columns="no_of_special_requests",
    values="Reservations"
)

print("\nReservation Counts by Market Segment and Special Requests:")
print(reservation_matrix.to_string())

fig = px.imshow(
    cancellation_matrix,
    text_auto=".1f",
    color_continuous_scale="Blues",
    aspect="auto",
    title="Cancellation Rates by Market Segment and Special Requests",
    labels={
        "x": "Number of Special Requests",
        "y": "Market Segment",
        "color": "Cancellation Rate (%)"
    },
    zmin=0,
    zmax=100
)

fig.update_layout(
    width=1050,
    height=550,
    template="plotly_white"
)

fig.show()

Cancellation Rates by Market Segment and Special Requests:
market_segment_type  no_of_special_requests  Reservations  Cancellations  Cancellation_Rate
           Aviation                       0           125             37              29.60
      Complementary                       0           167              0               0.00
      Complementary                       1           144              0               0.00
      Complementary                       2            50              0               0.00
      Complementary                       3            19              0               0.00
      Complementary                       4            11              0               0.00
          Corporate                       0          1613            209              12.96
          Corporate                       1           368             11               2.99
          Corporate                       2            28              0               0.00
          Corporate  

#### Key Observations
- **Online Reservations Without Special Requests:** Online bookings with no special requests have the highest observed cancellation rate at 60.71%, accounting for 5,616 cancellations across 9,251 reservations.
- **Declining Cancellation Rates With Additional Requests:** Within the Online segment, cancellation rates decrease from 60.71% with no requests to 24.38% with one request and 15.27% with two requests. Similar downward patterns appear in the Offline and Corporate segments.
- **Variation Across Market Segments:** Reservations without special requests have cancellation rates of 60.71% for Online, 31.12% for Offline, and 12.96% for Corporate bookings. No cancellations are recorded for reservations with three or more requests, although several segment combinations contain very few observations.

**Business Insights**

- **The particularly high cancellation rate among Online reservations without special requests identifies this combination as a major source of reservation uncertainty and a potentially valuable segment for cancellation-risk analysis.**
- **The consistent association between additional special requests and lower cancellation rates across major market segments suggests that combining guest engagement indicators with booking channels may provide more informative assessments of reservation reliability.**

## Time Series Analysis

#### Examining Daily Reservation Volumes and Rolling Trends

In [28]:
daily_reservations = (
    df.dropna(subset=["arrival_datetime"])
    .groupby("arrival_datetime")
    .size()
    .rename("Reservations")
    .asfreq("D", fill_value=0)
    .reset_index()
)

daily_reservations["Rolling_7D"] = (
    daily_reservations["Reservations"]
    .rolling(window=7, min_periods=7)
    .mean()
)

daily_reservations["Rolling_30D"] = (
    daily_reservations["Reservations"]
    .rolling(window=30, min_periods=30)
    .mean()
)

print("Daily Reservation Volume Summary:")
print(
    daily_reservations[
        ["Reservations", "Rolling_7D", "Rolling_30D"]
    ].describe().round(2).to_string()
)

print("\nDaily Reservation Coverage:")
print(f"Start Date: {daily_reservations['arrival_datetime'].min().date()}")
print(f"End Date: {daily_reservations['arrival_datetime'].max().date()}")
print(f"Total Days: {len(daily_reservations):,}")

fig = px.line(
    daily_reservations,
    x="arrival_datetime",
    y=["Reservations", "Rolling_7D", "Rolling_30D"],
    title="Daily Scheduled Reservations and Rolling Trends",
    labels={
        "arrival_datetime": "Scheduled Arrival Date",
        "value": "Number of Reservations",
        "variable": "Series"
    },
    template="plotly_white"
)

fig.update_layout(
    width=1100,
    height=550,
    hovermode="x unified",
    legend_title="Reservation Series",
    xaxis_title="Scheduled Arrival Date",
    yaxis_title="Number of Reservations"
)

fig.show()

Daily Reservation Volume Summary:
       Reservations  Rolling_7D  Rolling_30D
count        549.00      543.00       520.00
mean          66.01       66.15        67.44
std           43.56       31.99        28.76
min            1.00        6.43         9.83
25%           32.00       36.79        37.87
50%           61.00       72.86        72.87
75%           92.00       89.43        90.32
max          254.00      151.71       115.40

Daily Reservation Coverage:
Start Date: 2017-07-01
End Date: 2018-12-31
Total Days: 549


#### Key Observations
- **Daily Reservation Variability:** Scheduled reservation volumes average 66.01 reservations per day, with a standard deviation of 43.56 and daily counts ranging from 1 to 254.
- **Smoother Short-Term Trends:** The 7-day rolling average has a standard deviation of 31.99, compared with 43.56 for daily reservations, indicating reduced short-term variability.
- **Longer-Term Demand Patterns:** The 30-day rolling average exhibits further smoothing, with a standard deviation of 28.76 and values ranging from 9.83 to 115.40 reservations.

**Business Insights**

- **Substantial daily fluctuations in scheduled reservation volumes highlight the importance of forecasting methods that can account for short-term demand variability rather than relying solely on historical averages.**
- **The smoother 7-day and 30-day trends provide useful perspectives on underlying reservation patterns, supporting the investigation of historical trends and rolling statistics as potential forecasting features when constructed without future information.**

#### Examining Scheduled Reservation Volumes by Day of the Week

In [29]:
daily_reservations["Day_of_Week"] = (
    daily_reservations["arrival_datetime"].dt.day_name()
)

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekday_summary = (
    daily_reservations.groupby("Day_of_Week")
    .agg(
        Days=("Reservations", "count"),
        Total_Reservations=("Reservations", "sum"),
        Average_Reservations=("Reservations", "mean"),
        Median_Reservations=("Reservations", "median"),
        Std_Reservations=("Reservations", "std")
    )
    .reindex(weekday_order)
    .reset_index()
)

print("Scheduled Reservation Volumes by Day of the Week:")
print(weekday_summary.round(2).to_string(index=False))

fig = px.box(
    daily_reservations,
    x="Day_of_Week",
    y="Reservations",
    category_orders={"Day_of_Week": weekday_order},
    color="Day_of_Week",
    points="outliers",
    title="Distribution of Daily Scheduled Reservations by Day of the Week",
    labels={
        "Day_of_Week": "Scheduled Arrival Day",
        "Reservations": "Daily Scheduled Reservations"
    },
    template="plotly_white"
)

fig.update_layout(
    width=1050,
    height=550,
    showlegend=False,
    xaxis_title="Day of the Week",
    yaxis_title="Number of Scheduled Reservations"
)

fig.show()

Scheduled Reservation Volumes by Day of the Week:
Day_of_Week  Days  Total_Reservations  Average_Reservations  Median_Reservations  Std_Reservations
     Monday    79                5380                 68.10                 61.0             44.40
    Tuesday    78                4899                 62.81                 57.0             36.56
  Wednesday    78                5120                 65.64                 66.0             46.56
   Thursday    78                4544                 58.26                 56.0             36.71
     Friday    78                4741                 60.78                 59.0             42.20
   Saturday    79                5419                 68.59                 68.0             44.56
     Sunday    79                6135                 77.66                 75.0             50.80


#### Key Observations
- **Highest Sunday Reservation Volumes:** Sundays record the highest average daily reservations at 77.66, followed by Saturdays at 68.59 and Mondays at 68.10.
- **Lowest Thursday Reservation Volumes:** Thursdays have the lowest average at 58.26 reservations, approximately 25% below Sunday's average.
- **Greater Sunday Variability:** Sundays exhibit the highest standard deviation at 50.80 reservations, compared with 36.56 on Tuesdays, indicating greater fluctuations in scheduled reservation volumes.

**Business Insights**

- **Higher scheduled reservation volumes on Sundays suggest that day-of-week patterns may provide useful information for forecasting daily reservation demand and identifying recurring weekly fluctuations.**
- **The substantial variation in reservation volumes across weekdays highlights the importance of incorporating calendar-based features when developing forecasting models, while accounting for differences in daily demand variability.**

#### Examining Monthly Scheduled Reservation Patterns Across Years

In [30]:
daily_reservations["Year"] = (
    daily_reservations["arrival_datetime"].dt.year
)

daily_reservations["Month"] = (
    daily_reservations["arrival_datetime"].dt.month
)

monthly_summary = (
    daily_reservations.groupby(["Year", "Month"])
    .agg(
        Days=("Reservations", "count"),
        Total_Reservations=("Reservations", "sum"),
        Average_Daily_Reservations=("Reservations", "mean"),
        Std_Daily_Reservations=("Reservations", "std")
    )
    .reset_index()
)

monthly_summary["Month_Name"] = pd.to_datetime(
    monthly_summary["Month"],
    format="%m"
).dt.month_name()

monthly_summary["Year"] = monthly_summary["Year"].astype(str)

print("Monthly Scheduled Reservation Summary:")
print(monthly_summary.round(2).to_string(index=False))

fig = px.line(
    monthly_summary,
    x="Month_Name",
    y="Average_Daily_Reservations",
    color="Year",
    markers=True,
    category_orders={
        "Month_Name": [
            "January", "February", "March", "April",
            "May", "June", "July", "August",
            "September", "October", "November", "December"
        ]
    },
    title="Average Daily Scheduled Reservations by Month and Year",
    labels={
        "Month_Name": "Scheduled Arrival Month",
        "Average_Daily_Reservations": "Average Daily Reservations",
        "Year": "Year"
    },
    template="plotly_white",
    hover_data={
        "Days": True,
        "Total_Reservations": True,
        "Std_Daily_Reservations": ":.2f"
    }
)

fig.update_layout(
    width=1100,
    height=550,
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Average Daily Scheduled Reservations"
)

fig.show()

Monthly Scheduled Reservation Summary:
Year  Month  Days  Total_Reservations  Average_Daily_Reservations  Std_Daily_Reservations Month_Name
2017      7    31                 363                       11.71                   16.12       July
2017      8    31                1014                       32.71                   32.38     August
2017      9    30                1649                       54.97                   39.57  September
2017     10    31                1913                       61.71                   44.35    October
2017     11    30                 647                       21.57                   13.56   November
2017     12    31                 928                       29.94                   24.93   December
2018      1    31                1014                       32.71                   21.09    January
2018      2    28                1667                       59.54                   34.92   February
2018      3    31                2358               

C:\Users\user\AppData\Local\Temp\ipykernel_18896\1405814421.py:20: Pandas4Warning: Parsing integer or float values with a format in to_datetime is deprecated. In a future version these will be interpreted as epochs via the 'unit' keyword instead. Cast them to strings first to retain the current behavior.
  monthly_summary["Month_Name"] = pd.to_datetime(


#### Key Observations
- **Peak Monthly Reservation Demand:** October 2018 records the highest average daily reservation volume at 109.81, followed by June 2018 at 106.77 and September 2018 at 98.73.
- **Variation Across 2018:** Average daily reservations increase from 32.71 in January to 91.20 in April, with additional peaks in June and October before declining to 67.52 in December.
- **Differences Between Years:** For the overlapping July–December period, average daily reservation volumes are higher in 2018 than in 2017 for every month. However, these differences should not be interpreted as confirmed business growth without establishing comparable data coverage.

**Business Insights**

- **The substantial variation in monthly reservation volumes highlights the importance of capturing calendar-related demand patterns when developing daily reservation forecasting models.**
- **The elevated reservation volumes during June and October 2018 suggest periods of greater scheduled demand, supporting the use of monthly features while recognizing that the dataset's limited historical coverage restricts conclusions about recurring annual seasonality.**

#### Examining Daily Scheduled Reservations by Booking Status

In [31]:
daily_status = (
    df.dropna(subset=["arrival_datetime"])
    .groupby(["arrival_datetime", "booking_status"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        daily_reservations["arrival_datetime"],
        fill_value=0
    )
)

daily_status = daily_status.reindex(
    columns=["Not_Canceled", "Canceled"],
    fill_value=0
)

daily_status["Total_Reservations"] = daily_status.sum(axis=1)

daily_status["Cancellation_Rate"] = (
    daily_status["Canceled"]
    / daily_status["Total_Reservations"].replace(0, np.nan)
    * 100
)

print("Daily Reservation Summary by Booking Status:")
print(
    daily_status[
        ["Not_Canceled", "Canceled", "Total_Reservations", "Cancellation_Rate"]
    ].describe().round(2).to_string()
)

print("\nTotal Reservations by Booking Status:")
print(
    daily_status[
        ["Not_Canceled", "Canceled", "Total_Reservations"]
    ].sum().to_string()
)

daily_status_plot = daily_status.reset_index()

fig = px.area(
    daily_status_plot,
    x="arrival_datetime",
    y=["Not_Canceled", "Canceled"],
    title="Daily Scheduled Reservations by Final Booking Status",
    labels={
        "arrival_datetime": "Scheduled Arrival Date",
        "value": "Number of Reservations",
        "variable": "Booking Status"
    },
    template="plotly_white"
)

fig.update_layout(
    width=1100,
    height=550,
    hovermode="x unified",
    xaxis_title="Scheduled Arrival Date",
    yaxis_title="Number of Reservations",
    legend_title="Final Booking Status"
)

fig.show()

Daily Reservation Summary by Booking Status:
booking_status  Not_Canceled  Canceled  Total_Reservations  Cancellation_Rate
count                 549.00    549.00              549.00             549.00
mean                   44.37     21.64               66.01              28.40
std                    27.97     25.04               43.56              22.60
min                     0.00      0.00                1.00               0.00
25%                    26.00      3.00               32.00               9.52
50%                    42.00     15.00               61.00              25.58
75%                    59.00     31.00               92.00              41.67
max                   233.00    185.00              254.00             100.00

Total Reservations by Booking Status:
booking_status
Not_Canceled          24360
Canceled              11878
Total_Reservations    36238


#### Key Observations
- **Daily Reservation Composition:** Across 549 days, non-canceled reservations average 44.37 per day, while canceled reservations average 21.64, contributing to an overall daily average of 66.01 scheduled reservations.
- **Cancellation Variability:** Daily canceled reservations range from 0 to 185, with a standard deviation of 25.04, indicating substantial fluctuations in cancellation volumes.
- **Variation in Daily Cancellation Rates:** Daily cancellation rates average 28.40%, with a standard deviation of 22.60 percentage points and observed values ranging from 0% to 100%. These extremes may be influenced by days with relatively few reservations.

**Business Insights**

- **The substantial contribution of canceled reservations to scheduled reservation volumes highlights the importance of distinguishing total reservation demand from reservations that ultimately remain active.**
- **Considerable daily variation in cancellation activity suggests that final reservation outcomes can differ significantly across arrival dates, reinforcing the need to interpret scheduled demand forecasts separately from realized occupancy or confirmed arrivals.**

#### Examining Day-to-Day Changes in Scheduled Reservation Volumes

In [32]:
daily_reservations["Daily_Change"] = (
    daily_reservations["Reservations"].diff()
)

daily_reservations["Absolute_Daily_Change"] = (
    daily_reservations["Daily_Change"].abs()
)

print("Day-to-Day Reservation Change Summary:")
print(
    daily_reservations[
        ["Daily_Change", "Absolute_Daily_Change"]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    ).round(2).to_string()
)

print("\nAdditional Daily Change Statistics:")
print(
    f"Days With Increasing Reservations: "
    f"{(daily_reservations['Daily_Change'] > 0).sum():,}"
)
print(
    f"Days With Decreasing Reservations: "
    f"{(daily_reservations['Daily_Change'] < 0).sum():,}"
)
print(
    f"Days With No Change: "
    f"{(daily_reservations['Daily_Change'] == 0).sum():,}"
)

print(
    f"Days With Absolute Change Above 50: "
    f"{(daily_reservations['Absolute_Daily_Change'] > 50).sum():,}"
)

fig = px.histogram(
    daily_reservations.dropna(subset=["Daily_Change"]),
    x="Daily_Change",
    nbins=50,
    title="Distribution of Day-to-Day Changes in Scheduled Reservations",
    labels={
        "Daily_Change": "Change in Daily Scheduled Reservations",
        "count": "Number of Days"
    },
    template="plotly_white"
)

fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="red"
)

fig.update_layout(
    width=1050,
    height=550,
    xaxis_title="Day-to-Day Change in Reservations",
    yaxis_title="Number of Days",
    bargap=0.05,
    showlegend=False
)

fig.show()

Day-to-Day Reservation Change Summary:
       Daily_Change  Absolute_Daily_Change
count        548.00                 548.00
mean          -0.05                  30.79
std           44.82                  32.55
min         -208.00                   0.00
25%          -19.00                   7.00
50%           -0.50                  18.50
75%           18.00                  44.00
90%           51.00                  67.00
95%           69.30                  97.60
max          167.00                 208.00

Additional Daily Change Statistics:
Days With Increasing Reservations: 265
Days With Decreasing Reservations: 274
Days With No Change: 9
Days With Absolute Change Above 50: 111


#### Key Observations
- **Substantial Daily Fluctuations:** The average absolute day-to-day change is 30.79 reservations, with a median of 18.50, indicating considerable short-term variability in scheduled reservation volumes.
- **Large Reservation Changes:** Daily changes range from a decrease of 208 to an increase of 167 reservations, with 111 of 548 daily transitions (20.26%) exhibiting absolute changes exceeding 50 reservations.
- **Balanced Direction of Changes:** Reservation volumes increase on 265 days and decrease on 274 days, with only nine unchanged transitions. The mean daily change of -0.05 indicates little net directional movement across consecutive observations.

**Business Insights**

- **Frequent and occasionally extreme day-to-day changes in scheduled reservation volumes highlight the challenges of short-term demand forecasting and the importance of capturing temporal fluctuations.**
- **The substantial variability in daily reservation changes reinforces the need to evaluate forecasting models using appropriate error metrics and chronological validation, particularly during periods of abrupt demand changes.**

## EDA Summary

The exploratory data analysis of the StaySense Analytics dataset provides a comprehensive understanding of reservation characteristics, cancellation behavior, and temporal patterns in scheduled hotel reservation volumes.

### Key Findings

- **Reservation Characteristics:** The dataset contains 36,275 reservations, with Online bookings accounting for 63.99% of all reservations. Most reservations involve two guests, relatively short stays, and Room_Type 1.

- **Cancellation Behavior:** Approximately 32.76% of reservations are canceled. Cancellation rates vary considerably across booking lead times, market segments, room types, room prices, stay durations, and special-request activity.

- **Advance Booking Exposure:** Reservations made more than 180 days in advance have a cancellation rate of 73.89%, compared with 8.86% for reservations made within seven days, demonstrating a strong association between booking lead time and cancellation outcomes.

- **Market Segment Differences:** Online reservations exhibit higher cancellation rates than Corporate reservations, while Online bookings without special requests demonstrate particularly high cancellation exposure.

- **Guest Engagement Indicators:** Reservations involving special requests generally exhibit lower cancellation rates, suggesting that guest-specific booking characteristics provide useful information for understanding reservation outcomes.

- **Daily Reservation Variability:** Scheduled reservation volumes average 66.01 per day, with substantial fluctuations and occasional high-volume days. Daily changes frequently exceed 50 reservations, highlighting the complexity of short-term forecasting.

- **Weekly and Monthly Patterns:** Sundays exhibit the highest average scheduled reservation volumes, while monthly demand varies substantially, with particularly high volumes during June and October 2018.

- **Temporal Trends:** Rolling averages reveal smoother underlying reservation patterns, indicating that historical demand trends and calendar-based features may provide useful information for forecasting.

### Business Implications

- **Demand Forecasting:** Substantial daily fluctuations emphasize the need for forecasting models capable of capturing short-term changes in scheduled reservation volumes.

- **Calendar-Based Planning:** Differences across weekdays and months support the inclusion of calendar-related features when modeling reservation demand.

- **Reservation Uncertainty:** Cancellation patterns highlight the distinction between scheduled reservation volumes and reservations that ultimately remain active.

- **Feature Development:** Historical reservation counts, lagged observations, rolling statistics, and calendar variables represent potential forecasting features, provided they are constructed using information available at the forecast origin.

### Forecasting Considerations

The project focuses on forecasting daily scheduled reservation counts over a seven-day horizon using scheduled arrival dates.

The analysis covers 549 consecutive days from July 2017 through December 2018, after excluding 37 records with invalid arrival dates from date-dependent calculations.

Because the dataset contains retrospective reservation records rather than complete historical booking snapshots, final booking outcomes and scheduled reservation totals must not be assumed to have been available at earlier forecast origins.

Subsequent modeling will therefore require chronological data splitting, careful feature construction, and explicit acknowledgment of these information-availability limitations.

### Conclusion

The exploratory analysis identifies substantial daily variability, meaningful calendar patterns, and pronounced differences in reservation cancellation behavior.

These findings establish a foundation for developing and evaluating time-series forecasting models while highlighting the importance of preventing temporal leakage and interpreting predictions within the limitations of the available reservation data.